# 02 · Línea base

Evalúa el modelo **sin LoRA** (zero-shot con el mismo prompt) sobre el test set. Es el punto de comparación de todos los experimentos.

In [ ]:
# === Setup (en Colab se ejecuta solo; en local no hace nada) ===
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    !pip install -q -r requirements.txt
    !pip install -q -e ".[gpu]"

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, las salidas van a Drive para sobrevivir desconexiones.
if EN_COLAB:
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir  = config.drive_dir / "adaptadores"
    config.results_dir.mkdir(parents=True, exist_ok=True)
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name, "| resultados en:", config.results_dir)

## Cargar modelo base y evaluar en test

In [ ]:
from triaje.modelo import cargar_modelo_y_tokenizer
from triaje.data import cargar_split
from triaje.evaluate import evaluar

test = cargar_split(config.data_dir / "splits", "test")
model, tok = cargar_modelo_y_tokenizer(config, para_entrenar=False)
metricas = evaluar(model, tok, test, config,
                   guardar_predicciones_en=str(config.results_dir / "pred_base.jsonl"))
print("Esquema OK (post):", metricas.esquema_ok_post)
print("Exact match categóricos:", metricas.exact_match_categoricos)
print("Macro-F1:", metricas.macro_f1)
print("Tasa de sub-triaje:", metricas.tasa_subtriaje)